# Trial Batch Runner — Risk-Function Subtrials

Same shape as `trial_slow_batch_runner.ipynb`, but the per-tick target speed is computed by an arbitrary user-supplied function of **ground-truth risk** rather than a fixed multiplier:

```
v_target_kmh = SPEED_MULTIPLIER * risk_speed_fn(zone_speed_kmh, gt_risk)
```

`risk_speed_fn(v, r)` is your function. `v` is the live zone speed limit (km/h) read from CARLA each tick. `r` is the **ground-truth** scene risk computed on the fly by the simulator (no model involved). This isolates the risk-aware speed-control concept from any model quality concern — if the idea works at all, it will work here.

Set `SUBTRIAL_NAME` to one of `riskfn1`, `riskfn2`, `riskfn3` (the three slots the `trial_comparison` notebook plots in the bottom row of its 3×3 grid) and run the notebook once per function. Saves only the dataset.jsonl + summary.json, no images.

## 0 — CARLA Launch

In [ ]:
import subprocess

AUTO_LAUNCH_CARLA = True
CARLA_EXE = "CarlaUE4.exe"

if AUTO_LAUNCH_CARLA:
    subprocess.Popen(CARLA_EXE, shell=True)
    print(f"Launched {CARLA_EXE}")
else:
    print("AUTO_LAUNCH_CARLA is False. Assuming CARLA is already running.")

## 1 — Imports

In [ ]:
import sys
import time
from pathlib import Path

from MIREIA.config import Config
from MIREIA.simulation.trials import EgoTrialConfig, TrialDefinition, TrialRunner

## 2 — Discover Trials

In [ ]:
trials_root = Path(Config.PATH_TO_TRIALS)
all_trials = sorted(p.parent.name for p in trials_root.glob("*/trial.json") if p.is_file())

# Filtering (leave INCLUDE_PREFIXES empty to include every trial)
INCLUDE_PREFIXES: list[str] = []   # e.g. ["auto_"] to only run auto-generated trials
EXCLUDE_TRIALS:   list[str] = []   # exact trial names to skip

if INCLUDE_PREFIXES:
    selected_trial_names = [n for n in all_trials if any(n.startswith(p) for p in INCLUDE_PREFIXES)]
else:
    selected_trial_names = list(all_trials)
selected_trial_names = [n for n in selected_trial_names if n not in EXCLUDE_TRIALS]

print(f"Found {len(all_trials)} trial(s); {len(selected_trial_names)} selected to run:")
for n in selected_trial_names:
    print(f"  - {n}")

## 3 — Risk Function & Subtrial Config

`risk_speed_fn(v, r)` returns the desired target speed (km/h) for that tick.

- `v` — live zone speed limit (km/h), read from CARLA every tick.
- `r` — ground-truth scene risk for the current ego position (float, typically 0..~1; can be `None` if the oracle fails momentarily).

Pick a `SUBTRIAL_NAME` from `riskfn1` / `riskfn2` / `riskfn3` so the run lands in the 3rd row of the `trial_comparison` grid. Edit the function below and re-run the notebook to test more variants. The preview shows what `v'` your function returns at a few `(v, r)` samples — sanity-check the shape before launching the batch.

In [ ]:
SUBTRIAL_NAME       = "riskfn1"   # one of: "riskfn1", "riskfn2", "riskfn3"
SPEED_MULTIPLIER    = 1.0         # final v' = SPEED_MULTIPLIER * risk_speed_fn(v, r)
MAX_STEPS_PER_TRIAL = 6000        # cap; raise if your function makes runs slow

# Define the risk-aware speed function here.  Edit freely between runs.
# Convention: r near 0 -> safe -> drive at full v; r near 1 -> risky -> slow down.
def risk_speed_fn(v: float, r: float | None) -> float:
    if r is None:
        return v
    # Linearly slow the car from v at r=0 to 0.3*v at r=1 (clamped).
    r_clipped = max(0.0, min(1.0, float(r)))
    return v * (1.0 - 0.7 * r_clipped)

# Smoke-preview the function at a few (zone_limit, risk) samples.
print(f"risk_speed_fn preview  ({SUBTRIAL_NAME}, SPEED_MULTIPLIER={SPEED_MULTIPLIER}):")
print(f"  {'v_kmh':>7s} {'r':>5s} -> {'v_out_kmh':>9s}")
for v in (30.0, 50.0, 90.0):
    for r in (None, 0.0, 0.25, 0.5, 0.75, 1.0):
        out = SPEED_MULTIPLIER * risk_speed_fn(v, r)
        r_disp = "None" if r is None else f"{r:.2f}"
        print(f"  {v:7.1f} {r_disp:>5s} -> {out:9.2f}")

ego_cfg = EgoTrialConfig(
    name=SUBTRIAL_NAME,
    ego_blueprint="vehicle.lincoln.mkz_2020",
    # target_speed_kmh=None  -> controller uses live zone speed limit as `v`.
    speed_multiplier=SPEED_MULTIPLIER,
    use_vehicle_camera_defaults=True,
    controller_mode="behavior_agent",
    controller_behavior="normal",
)

runner = TrialRunner(verbose=False)
print(f"\nRisk-fn ego config ready: subtrial='{SUBTRIAL_NAME}', SPEED_MULTIPLIER={SPEED_MULTIPLIER}")

## 4 — Run Each Trial

In [ ]:
all_summaries = []
failed: list[tuple[str, str]] = []

# --- Run-time accounting (sim vs wall, cap meaning) ----------------------------
# step  = one CARLA world.tick() = one fixed_delta seconds of simulated time.
# image_stride controls JSONL cadence, NOT loop iteration count.
# The loop exits early when controller.done() — ego reached the route end (within
# arrival_threshold_m = 3 m). max_steps is the cap; functions that slow the car
# more cover the same distance in MORE sim time, so they need a higher cap.
fixed_delta_s    = float(Config.SIM_FIXED_DELTA_SECONDS)
stride_ticks     = int(Config.RECORD_EVERY_N_TICKS)
sim_time_cap_s   = MAX_STEPS_PER_TRIAL * fixed_delta_s
max_jsonl_frames = MAX_STEPS_PER_TRIAL // stride_ticks
record_hz        = 1.0 / (fixed_delta_s * stride_ticks)

print("Run configuration:")
print(f"  subtrial_name:        '{SUBTRIAL_NAME}'")
print(f"  SPEED_MULTIPLIER:     {SPEED_MULTIPLIER:.3f}")
print(f"  risk source:          ground truth (wm.get_risk() per tick)")
print(f"  fixed_delta:          {fixed_delta_s:.3f} s/tick  ({1/fixed_delta_s:.1f} Hz physics)")
print(f"  image_stride:         {stride_ticks} ticks/frame  ({record_hz:.2f} Hz JSONL records)")
print(f"  max_steps_per_trial:  {MAX_STEPS_PER_TRIAL}  (cap = {sim_time_cap_s:.1f} s sim time, "
      f"<= {max_jsonl_frames} JSONL frames)")
print(f"  finished=True   -> controller reported done before the cap (good)")
print(f"  finished=False  -> hit max_steps cap (raise MAX_STEPS_PER_TRIAL or make risk_speed_fn faster)")

_progress_state: dict = {"trial_started": 0.0}

def _format_secs(s: float) -> str:
    if s < 60:
        return f"{s:5.1f}s"
    m, sec = divmod(s, 60)
    return f"{int(m):02d}:{int(sec):02d}"

def _on_progress(step: int, max_steps: int) -> None:
    now = time.time()
    last = _progress_state.get("last_print", 0.0)
    if now - last < 0.2:
        return
    _progress_state["last_print"] = now
    elapsed = now - _progress_state["trial_started"]
    rate = step / elapsed if elapsed > 0 else 0.0
    pct = 100.0 * step / max_steps if max_steps else 0.0
    eta_to_cap = (max_steps - step) / rate if rate > 0 else float("inf")
    speedup = (step * fixed_delta_s) / elapsed if elapsed > 0 else 0.0
    sys.stdout.write(
        f"\r    step {step:5d}/{max_steps} ({pct:5.1f}% of cap)  "
        f"wall={_format_secs(elapsed)}  "
        f"rate={rate:5.1f} t/s  "
        f"speedup={speedup:4.2f}x  "
        f"eta_to_cap={_format_secs(eta_to_cap) if rate > 0 else '  inf'}    "
    )
    sys.stdout.flush()

batch_started = time.time()
for idx, trial_name in enumerate(selected_trial_names, start=1):
    print()
    print(f">>> [{idx}/{len(selected_trial_names)}] {trial_name}", flush=True)
    _progress_state["trial_started"] = time.time()
    _progress_state["last_print"] = 0.0
    try:
        trial = TrialDefinition.load(trial_name)
        summary = runner.run_subtrial(
            trial=trial,
            ego_cfg=ego_cfg,
            max_steps=MAX_STEPS_PER_TRIAL,
            image_stride=Config.RECORD_EVERY_N_TICKS,
            store_rgb_images=False,
            store_topdown_images=False,
            store_risk_frame_images=False,
            store_static_risk_map=False,
            draw_debug_every_tick=False,
            risk_speed_fn=risk_speed_fn,
            use_ground_truth_risk=True,
            progress_callback=_on_progress,
            progress_every_n_steps=25,
        )
        elapsed = time.time() - _progress_state["trial_started"]
        sim_seconds = summary.num_frames * fixed_delta_s * stride_ticks
        speedup = sim_seconds / elapsed if elapsed > 0 else float("inf")
        sys.stdout.write("\r" + " " * 130 + "\r")
        sys.stdout.flush()
        print(
            f"    ok in {elapsed:6.1f}s wall  "
            f"frames={summary.num_frames:4d}  "
            f"sim_t={sim_seconds:6.1f}s  "
            f"speedup={speedup:5.2f}x  "
            f"dist={summary.traveled_m:7.1f}m  "
            f"risk_auc={summary.risk_auc:.3f}  "
            f"finished={summary.finished}"
        )
        all_summaries.append(summary)
    except Exception as e:
        elapsed = time.time() - _progress_state["trial_started"]
        sys.stdout.write("\r" + " " * 130 + "\r")
        sys.stdout.flush()
        print(f"    FAILED in {elapsed:.1f}s: {type(e).__name__}: {e}")
        failed.append((trial_name, f"{type(e).__name__}: {e}"))

batch_elapsed = time.time() - batch_started
print()
print(f"Batch done in {batch_elapsed:.1f}s (ok={len(all_summaries)}, failed={len(failed)})")

## 5 — Summary

In [ ]:
if not all_summaries and not failed:
    print("No trials were run.")
else:
    print(f"=== Successful runs ({len(all_summaries)}) ===")
    print(
        f"  {'trial_name':50s} {'frames':>6s} {'dist_m':>9s} "
        f"{'risk_auc':>9s} {'risk/m':>9s}  finished  run_path"
    )
    for s in all_summaries:
        print(
            f"  {s.trial_name:50s} {s.num_frames:6d} {s.traveled_m:9.1f} "
            f"{s.risk_auc:9.3f} {s.risk_per_meter:9.5f}  {str(s.finished):>8s}  {s.run_path}"
        )

    if failed:
        print()
        print(f"=== Failed runs ({len(failed)}) ===")
        for name, err in failed:
            print(f"  {name}: {err}")